In [1]:
from einops import rearrange, repeat, reduce, pack, unpack

In [5]:
import torch
from vector_quantize_pytorch import VectorQuantize

# toy setup
vq = VectorQuantize(
            dim=16,               # latent dim
            codebook_size=8,      # number of embeddings
            decay=0.8,            # EMA update
            # EMA update related params
            # use_cosine_sim=True,
            # ema_update=True, # if true, learnable related stuff is ignored
            # manual_ema_update=False,
            # heads=1,
            # manual_in_place_optimizer_update=False, # must be false if ema_update is true
            # learnable_codebook=False,
            # Learnable update related params
            use_cosine_sim=False,
            learnable_codebook=True,
            straight_through=False,
            rotation_trick=True,
            in_place_codebook_optimizer=None,
            manual_in_place_optimizer_update=False,
            ema_update=False,    
            eps=0.00001,
            kmeans_init=False,
            kmeans_iters=10,
            sync_kmeans=True,
            sample_codebook_temp=0.0,
            # commitment loss related params
            commitment_weight=0.0,
            commitment_use_cross_entropy_loss=False,
            # codebook diversity loss related params
            codebook_diversity_loss_weight=0.0,
            codebook_diversity_temperature=100.0,
            # orthogonal loss related params
            orthogonal_reg_weight=0.0,
            orthogonal_reg_max_codes=None, # apply to a random subset of codes
            orthogonal_reg_active_codes_only=False # apply only to actively used codes in the current batch    
        )

x = torch.randn(2, 16)  # encoder output

# ---- training mode ----
vq.train()
quantized_train, indices, _ = vq(x)  # forward returns quantized, indices, loss terms
print("quantized_train == codes ?", torch.allclose(quantized_train, vq.codebook[indices]))
print("quantized_train == codes ?", torch.allclose(quantized_train, vq.get_codes_from_indices(indices)))
print(f"{quantized_train.shape=} | {vq.codebook[indices].shape=} | {vq.get_codes_from_indices(indices).shape}")
# False, because STE adds (x - codes).detach() to let grads flow
print(f"{indices.shape=}")

# ---- eval mode ----
vq.eval()
quantized_eval, indices, _ = vq(x)
print("quantized_eval == codes ?", torch.allclose(quantized_eval, vq.codebook[indices]))
print("quantized_eval == codes ?", torch.allclose(quantized_eval, vq.get_codes_from_indices(indices)))
print(f"{quantized_eval.shape=} | {vq.codebook[indices].shape=} | {vq.get_codes_from_indices(indices).shape}")
print(f"{indices.shape=}")
# True, because forward reduces to pure codebook lookup

quantized_train == codes ? True
quantized_train == codes ? True
quantized_train.shape=torch.Size([2, 16]) | vq.codebook[indices].shape=torch.Size([2, 16]) | torch.Size([2, 16])
indices.shape=torch.Size([2])
quantized_eval == codes ? True
quantized_eval == codes ? True
quantized_eval.shape=torch.Size([2, 16]) | vq.codebook[indices].shape=torch.Size([2, 16]) | torch.Size([2, 16])
indices.shape=torch.Size([2])


In [39]:
import torch
from vector_quantize_pytorch import VectorQuantize

# toy setup
vq = VectorQuantize(
            dim=16,               # latent dim
            codebook_size=8,      # number of embeddings
            decay=0.8,            # EMA update
            # EMA update related params
            use_cosine_sim=True,
            ema_update=True, # if true, learnable related stuff is ignored
            manual_ema_update=False,
            heads=2
        )

x = torch.randn(2, 16)  # encoder output

# ---- training mode ----
vq.train()
quantized_train, indices, _ = vq(x)  # forward returns quantized, indices, loss terms
print("quantized_train == codes ?", torch.allclose(quantized_train, vq.codebook[indices]))
print("quantized_train == codes ?", torch.allclose(quantized_train, vq.get_codes_from_indices(indices)))
# print("quantized_train == codes ?", torch.allclose(quantized_train, vq.get_output_from_indices(indices)))
print("quantized_train == codes ?", torch.allclose(vq.codebook[indices], vq.get_codes_from_indices(indices)))
print(f"{quantized_train.shape=} | {vq.codebook[indices].shape=} | {vq.get_codes_from_indices(indices).shape=}")
# False, because STE adds (x - codes).detach() to let grads flow
print(f"{indices.shape=}")

# ---- eval mode ----
vq.eval()
quantized_eval, indices, _ = vq(x)
print("quantized_eval == codes ?", torch.allclose(quantized_eval, vq.codebook[indices]))
print("quantized_eval == codes ?", torch.allclose(quantized_eval, vq.get_codes_from_indices(indices)))
print("quantized_eval == codes ?", torch.allclose(quantized_eval, vq.get_output_from_indices(indices)))
print("quantized_eval == codes ?", torch.allclose(vq.codebook[indices], vq.get_codes_from_indices(indices)))
print(f"{quantized_eval.shape=} | {vq.codebook[indices].shape=} | {vq.get_codes_from_indices(indices).shape=}")
# True, because forward reduces to pure codebook lookup
print(f"{indices.shape=}")

quantized_train == codes ? False
quantized_train == codes ? False
quantized_train == codes ? True
quantized_train.shape=torch.Size([2, 16]) | vq.codebook[indices].shape=torch.Size([2, 2, 16]) | vq.get_codes_from_indices(indices).shape=torch.Size([2, 2, 16])
indices.shape=torch.Size([2, 2])
quantized_eval == codes ? False
quantized_eval == codes ? False


RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x16 and 32x16)

In [34]:
quantized_eval

tensor([[-1.2596e-01,  2.3400e-01, -3.0839e-01, -2.3061e-01, -4.6216e-02,
         -2.3621e-02, -2.0050e-01, -1.8160e-01, -3.2487e-01,  2.4608e-01,
          1.1358e-02, -2.5572e-01, -1.6619e-03,  1.6656e-01, -1.5357e-02,
          1.8044e-01],
        [-7.3723e-02,  8.2078e-02, -1.5874e-01, -1.1091e-01, -1.8046e-01,
         -1.0813e-01, -1.7202e-04,  4.5958e-02,  4.3993e-02,  2.6668e-01,
         -3.3896e-01,  1.6165e-01,  8.5134e-02, -9.4851e-02, -2.4742e-01,
          2.4117e-01]], grad_fn=<ViewBackward0>)

In [33]:
vq.codebook[indices]

tensor([[[-0.0671,  0.1625,  0.2647, -0.3752, -0.1485,  0.0297, -0.3290,
          -0.3572, -0.4186, -0.0504,  0.1511, -0.0260,  0.4459, -0.0432,
          -0.2433,  0.1956],
         [-0.0671,  0.1625,  0.2647, -0.3752, -0.1485,  0.0297, -0.3290,
          -0.3572, -0.4186, -0.0504,  0.1511, -0.0260,  0.4459, -0.0432,
          -0.2433,  0.1956]],

        [[-0.2295, -0.1026, -0.3836,  0.0703,  0.1650,  0.2635,  0.3799,
           0.3503, -0.0452, -0.1813,  0.3150, -0.3756,  0.2014, -0.1656,
           0.0598, -0.2726],
         [-0.0610,  0.0491,  0.2572, -0.1749, -0.0676,  0.1822,  0.1654,
          -0.2465,  0.1781, -0.3764,  0.3160,  0.4762,  0.2869, -0.3655,
           0.1535,  0.1782]]])

In [36]:
vq.get_output_from_indices(indices)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x16 and 32x16)

In [16]:
indices.shape

torch.Size([2, 2])

In [17]:
indices

tensor([[6, 7],
        [4, 3]])

In [30]:
embed_ind = rearrange(indices, 'h b n -> b n h', h = 2)

EinopsError:  Error while processing rearrange-reduction pattern "h b n -> b n h".
 Input tensor shape: torch.Size([2, 2]). Additional info: {'h': 2}.
 Wrong shape: expected 3 dims. Received 2-dim tensor.